# Chapter 21 — Beyond the Book: Production DSPy Systems (Appendix)

**Book alignment:** DSPy From First Principles, Chapter 21

**Question this notebook isolates:** Do the production kernels hold on fixtures: budget gate caps calls, champion moves only on improvement, parser normalizes all three shapes?


In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

spec = importlib.util.spec_from_file_location(
    "ch21_beyond_run", str(EXP_ROOT / "ch21_beyond_book" / "run.py")
)
kernels = importlib.util.module_from_spec(spec)
sys.modules["ch21_beyond_run"] = kernels
spec.loader.exec_module(kernels)
print("kernels loaded: UCT selection, budget cache, champion A/B, CBR parser (no LM, no DSPy import)")


## Reasoning search with a budget

Every generation call checks the call ceiling first, cache hits cost nothing, and unvisited branches score infinite so each option is tried before statistics dominate.


In [ ]:
uct = kernels.run_uct(parent_visits=13, ucb_weight=1.41)
budget = kernels.run_budget_cache(max_lm_calls=12, tail_len=6)
for child in uct["children"]:
    print(f"child {child['id']}: visits={child['visits']} reward={child['reward']} uct={child['uct']:.4f}")
print("selection order:", uct["selection_order"])
print("calls used:", budget["calls_used"], "| within budget:", budget["within_budget"], "| replay cost zero:", budget["repeat_tail_cost_zero"])


In [ ]:
assert uct["selection_order"] == ["a", "c", "b"]
assert uct["claims"]["unvisited_first"] is True
assert budget["within_budget"] is True and budget["calls_used"] <= 12
assert budget["repeat_tail_cost_zero"] is True
print("unvisited -> under-explored -> leader; identical tails pay once")


## Memory that refuses to regress

The champion is replaced only on measured improvement under a locked seed. An epsilon floor stops zero from promoting, but only a practical margin stops noise from promoting.


In [ ]:
ab = kernels.run_ab(delta_eps=1e-6, margin_demo=0.01)
for event in ab["events"]:
    print(f"base={event['q_base']:.7f} cbr={event['q_cbr']:.7f} improved={event['improved']!s:5s} promoted={event['promoted']!s:5s} clears_margin={event['clears_margin_0_01']!s:5s}")
print("final champion quality:", ab["final_champion_quality"])


In [ ]:
events = ab["events"]
assert events[0]["promoted"] is False
assert events[1]["promoted"] is False
assert events[2]["promoted"] is True and events[2]["clears_margin_0_01"] is False
assert events[3]["promoted"] is True
assert events[4]["promoted"] is False
assert ab["final_champion_quality"] == 0.78
assert ab["ties_hold_champion"] is True and ab["regression_holds_champion"] is True
print("epsilon stops zero; only a real margin stops noise")


## A small CBR module that earns its place

The declared `list[str]` output does not guarantee list-shaped arrivals, so the parser normalizes lists, numbered text, and unexpected shapes with a cap and a log line for the shape nobody predicted.


In [ ]:
parsed = kernels.run_parse()
for name, case in parsed["cases"].items():
    print(f"{name:18s} shape={case['shape']:22s} parsed={case['parsed']}")


In [ ]:
assert parsed["list_normalized"] is True
assert parsed["text_normalized"] is True
assert parsed["unexpected_logged_not_dropped_silently"] is True
print("parser outranks the type declaration; unknown shapes are recorded, not dropped")


## What we earned

This appendix changes no conclusion in Chapters 15-20. It shows the book's machinery running one level up: a search with a written-down budget, a memory whose champion moves only on a margin-clearing improvement, and a small module whose parser does the work its type signature merely declares.

The pattern across all three kernels is the book's thesis at production scale: the language model generates, and deterministic software decides what that generation is allowed to mean.
